# Setup and configuration

In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

In [2]:
from pathlib import Path

ROOT_DIR = Path.cwd()

DATA_DIR = ROOT_DIR / "data"
USER_DATA_DIR = DATA_DIR / "User Data"
USER_TRADES_DIR = DATA_DIR / "User Trades"

OUTPUT_DIR = ROOT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Raw file audit

## Header tokens

In [4]:
ROSTER_HEADER_TOKENS = {
    "account",
    "email",
    "ip_address",
    "ip address",
    "telegram_username",
    "challenge_type_id",
}

TRADE_HEADER_TOKENS = {
    "accountid",
    "orderid",
    "opendatetime",
    "closedatetime",
    "profit",
    "reverseprofit",
    "netprofit",
    "commission",
    "swap",
    "amount",
    "openprice",
    "closeprice",
    "slprice",
    "tpprice",
    "side",
    "currency",
    "opentradecrossprice",
    "closetradecrossprice",
    "usergroupid",
    "campaignid",
}

## Audit function

In [5]:
def audit_file(
    path: str | Path,
    header_tokens: set[str],
    min_matches: int = 2,
) -> dict:
    """Scans a CSV or Excel file for embedded duplicate-header rows.

    The file is loaded without treating the first row as a header. Each
    subsequent row is then checked for values that match known header tokens.
    Rows with at least `min_matches` matching tokens are flagged as possible
    embedded duplicate headers.

    Args:
        path: Path to the CSV or Excel file to audit.
        header_tokens: Normalized header values expected in the file.
        min_matches: Minimum number of token matches required to flag a row.
            Defaults to 2.

    Returns:
        A dictionary containing:
            - `file`: File name.
            - `total_rows`: Number of data rows excluding the true header.
            - `echo_rows`: Number of possible embedded-header rows.
            - `echo_row_numbers`: Spreadsheet-style row numbers of the
              suspicious rows.
            - `error`: Error message if the file could not be read, otherwise
              `None`.
    """
    path = Path(path)

    try:
        if path.suffix.lower() == ".xlsx":
            raw = pd.read_excel(path, header=None)
        elif path.suffix.lower() == ".csv":
            raw = pd.read_csv(path, header=None)
        else:
            return {
                "file": path.name,
                "total_rows": None,
                "echo_rows": None,
                "echo_row_numbers": [],
                "error": f"Unsupported file type: {path.suffix}",
            }

    except Exception as e:
        return {
            "file": path.name,
            "total_rows": None,
            "echo_rows": None,
            "echo_row_numbers": [],
            "error": str(e),
        }

    total_rows = max(len(raw) - 1, 0)
    echo_row_numbers = []

    for row_position in range(1, len(raw)):
        row_values = {
            str(value).strip().lower()
            for value in raw.iloc[row_position].tolist()
            if pd.notna(value)
        }

        n_matches = len(row_values.intersection(header_tokens))

        if n_matches >= min_matches:
            echo_row_numbers.append(row_position + 1)

    return {
        "file": path.name,
        "total_rows": total_rows,
        "echo_rows": len(echo_row_numbers),
        "echo_row_numbers": echo_row_numbers,
        "error": None,
    }

## Discover source files function

In [ ]:
def discover_source_files(
    user_data_dir: str | Path,
    user_trades_dir: str | Path,
) -> pd.DataFrame:
    """Discovers trader and trade files from their respective directories.

    Args:
        user_data_dir: Directory containing account or participant files.
        user_trades_dir: Directory containing XAUUSD trade files.

    Returns:
        A DataFrame containing the file path, file name, file type,
        extension, and size of each discovered source file.

    Raises:
        FileNotFoundError: If either source directory does not exist.
    """
    user_data_dir = Path(user_data_dir)
    user_trades_dir = Path(user_trades_dir)

    if not user_data_dir.exists():
        raise FileNotFoundError(
            f"User Data directory was not found: "
            f"{user_data_dir.resolve()}"
        )

    if not user_trades_dir.exists():
        raise FileNotFoundError(
            f"User Trades directory was not found: "
            f"{user_trades_dir.resolve()}"
        )

    records = []

    folder_mapping = {
        "trader": user_data_dir,
        "trades": user_trades_dir,
    }

    for file_type, folder in folder_mapping.items():
        for path in folder.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in {".csv", ".xlsx"}:
                continue

            records.append({
                "path": path,
                "file": path.name,
                "file_type": file_type,
                "extension": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            })

    if not records:
        return pd.DataFrame(
            columns=[
                "path",
                "file",
                "file_type",
                "extension",
                "size_bytes",
            ]
        )

    return (
        pd.DataFrame(records)
        .sort_values(["file_type", "file"])
        .reset_index(drop=True)
    )

In [69]:
source_files = discover_source_files(
    user_data_dir=USER_DATA_DIR,
    user_trades_dir=USER_TRADES_DIR,
)

print("Total source files:", len(source_files))
print()
print(source_files["file_type"].value_counts())

display(source_files.head(10))

Total source files: 68

file_type
roster    34
trades    34
Name: count, dtype: int64


,path,file,file_type,extension,size_bytes
0,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,roster,.xlsx,29392
1,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 34 Data 03 Mar 2026 Traders only (1D)...,roster,.xlsx,20980
2,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 35 Data 06 Mar 2026 Traders only (1D)...,roster,.xlsx,28800
3,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 36 Data 09 Mar 2026 Traders only (1D)...,roster,.xlsx,25426
4,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 37 Data 13 Mar 2026 Traders only (1D)...,roster,.xlsx,27312
5,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 38 Data 17 Mar 2026 Traders only (1D)...,roster,.xlsx,29139
6,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 39 Data 20 Mar 2026 Traders only (1D)...,roster,.xlsx,22956
7,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 40 Data 24 Mar 2026 Traders only (1D)...,roster,.xlsx,24701
8,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 41 Data 27 Mar 2026 Traders only (1D)...,roster,.xlsx,18457
9,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,Campaign 42 Data 31 Mar 2026 Traders only (1D)...,roster,.xlsx,28989


## Audit all discovered files

In [8]:
audit_records = []

for file_record in source_files.itertuples(index=False):
    if file_record.file_type == "roster":
        header_tokens = ROSTER_HEADER_TOKENS
    else:
        header_tokens = TRADE_HEADER_TOKENS

    audit_result = audit_file(
        path=file_record.path,
        header_tokens=header_tokens,
        min_matches=2,
    )

    audit_result["file_type"] = file_record.file_type
    audit_result["path"] = file_record.path

    audit_records.append(audit_result)

audit_df = pd.DataFrame(audit_records)

In [9]:
audit_summary = pd.Series({
    "files_audited": len(audit_df),
    "files_with_read_errors": audit_df["error"].notna().sum(),
    "files_with_embedded_headers": (
        audit_df["echo_rows"].fillna(0) > 0
    ).sum(),
    "embedded_header_rows_found": (
        audit_df["echo_rows"].fillna(0).sum()
    ),
})

display(audit_summary.to_frame("value"))

,value
files_audited,68
files_with_read_errors,0
files_with_embedded_headers,1
embedded_header_rows_found,4


In [10]:
audit_issues = audit_df.loc[
    audit_df["error"].notna()
    | audit_df["echo_rows"].fillna(0).gt(0),
    [
        "file_type",
        "file",
        "total_rows",
        "echo_rows",
        "echo_row_numbers",
        "error",
    ],
]

display(audit_issues)

,file_type,file,total_rows,echo_rows,echo_row_numbers,error
24,roster,Campaign 57 Data 02 June 2026 Traders only (1D...,504,4,"[102, 203, 304, 405]",None


In [11]:
audit_df.to_csv(
    OUTPUT_DIR / "raw_file_audit.csv",
    index=False,
)

# Load and standardize the campaign files

## Parse campaign metadata

In [12]:
FILENAME_PATTERN = re.compile(
    r"Campaign[_\s]*(\d+)"
    r"[_\s]*Data[_\s]*"
    r"(\d{1,2}[_\s]*[A-Za-z]+[_\s]*\d{4})"
    r".*?"
    r"(Traders[\s_]*only|XAUUSD[\s_]*only)",
    re.IGNORECASE,
)

In [13]:
def parse_campaign_date(date_string: str) -> pd.Timestamp:
    """Parses a campaign date using abbreviated or full month names.

    Args:
        date_string: Campaign date extracted from a filename.

    Returns:
        A parsed pandas Timestamp. Returns `pd.NaT` if none of the
        supported formats can parse the date.
    """
    supported_formats = [
        "%d %b %Y",  # Example: 02 Jun 2026
        "%d %B %Y",  # Example: 02 June 2026
    ]

    for date_format in supported_formats:
        parsed_date = pd.to_datetime(
            date_string,
            format=date_format,
            errors="coerce",
        )

        if pd.notna(parsed_date):
            return parsed_date

    return pd.NaT

def parse_filename(path: str | Path) -> dict:
    """Extracts campaign metadata from a roster or trade filename.

    The function attempts to extract the campaign ID, campaign date, and
    source-file type from the filename. If the complete expected pattern is
    unavailable, it applies fallback rules for the campaign ID and file type.

    Args:
        path: Path to a roster or XAUUSD trade file.

    Returns:
        A dictionary containing:
            - `campaign_id`: Extracted campaign number, or `None`.
            - `campaign_date`: Parsed pandas Timestamp, or `NaT`.
            - `file_type`: Either `"roster"`, `"trades"`, or `None`.
            - `filename_matched`: Whether the complete filename pattern matched.
            - `parse_warning`: Description of any parsing issue, otherwise
              `None`.
    """
    path = Path(path)
    filename = path.name

    match = FILENAME_PATTERN.search(filename)

    if match:
        campaign_id = int(match.group(1))

        date_string = re.sub(
            r"[_\s]+",
            " ",
            match.group(2).strip(),
        )

        raw_file_type = match.group(3).lower()

        file_type = (
            "roster"
            if "traders" in raw_file_type
            else "trades"
        )

        campaign_date = parse_campaign_date(date_string)

        parse_warning = None

        if pd.isna(campaign_date):
            parse_warning = (
                f"Campaign date could not be parsed from "
                f"'{date_string}'."
            )

        return {
            "campaign_id": campaign_id,
            "campaign_date": campaign_date,
            "file_type": file_type,
            "filename_matched": True,
            "parse_warning": parse_warning,
        }

    campaign_match = re.search(
        r"Campaign[_\s]*(\d+)",
        filename,
        re.IGNORECASE,
    )

    campaign_id = (
        int(campaign_match.group(1))
        if campaign_match
        else None
    )

    filename_lower = filename.lower()
    parent_name_lower = path.parent.name.lower()

    if (
        "traders" in filename_lower
        or parent_name_lower == "user data"
    ):
        file_type = "roster"

    elif (
        "xauusd" in filename_lower
        or parent_name_lower == "user trades"
    ):
        file_type = "trades"

    else:
        file_type = None

    return {
        "campaign_id": campaign_id,
        "campaign_date": pd.NaT,
        "file_type": file_type,
        "filename_matched": False,
        "parse_warning": (
            "The filename did not match the complete expected pattern. "
            "Fallback extraction was applied."
        ),
    }

## Validate filename parsing

### Check parsing quality

In [14]:
filename_metadata_records = []

for file_record in source_files.itertuples(index=False):
    metadata = parse_filename(file_record.path)

    filename_metadata_records.append({
        "file": file_record.file,
        "path": file_record.path,
        "discovered_file_type": file_record.file_type,
        **metadata,
    })

filename_metadata_df = pd.DataFrame(
    filename_metadata_records
)

display(filename_metadata_df.head(10))

,file,path,discovered_file_type,campaign_id,campaign_date,file_type,filename_matched,parse_warning
0,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,33,2026-02-24,roster,True,None
1,Campaign 34 Data 03 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,34,2026-03-03,roster,True,None
2,Campaign 35 Data 06 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,35,2026-03-06,roster,True,None
3,Campaign 36 Data 09 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,36,2026-03-09,roster,True,None
4,Campaign 37 Data 13 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,37,2026-03-13,roster,True,None
5,Campaign 38 Data 17 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,38,2026-03-17,roster,True,None
6,Campaign 39 Data 20 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,39,2026-03-20,roster,True,None
7,Campaign 40 Data 24 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,40,2026-03-24,roster,True,None
8,Campaign 41 Data 27 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,41,2026-03-27,roster,True,None
9,Campaign 42 Data 31 Mar 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,roster,42,2026-03-31,roster,True,None


In [15]:
parsing_summary = pd.Series({
    "total_files": len(filename_metadata_df),
    "complete_filename_matches": (
        filename_metadata_df["filename_matched"].sum()
    ),
    "fallback_matches": (
        ~filename_metadata_df["filename_matched"]
    ).sum(),
    "missing_campaign_ids": (
        filename_metadata_df["campaign_id"].isna().sum()
    ),
    "missing_campaign_dates": (
        filename_metadata_df["campaign_date"].isna().sum()
    ),
    "file_type_mismatches": (
        filename_metadata_df["discovered_file_type"]
        != filename_metadata_df["file_type"]
    ).sum(),
})

display(parsing_summary.to_frame("value"))

,value
total_files,68
complete_filename_matches,68
fallback_matches,0
missing_campaign_ids,0
missing_campaign_dates,0
file_type_mismatches,0


### Show problematic files

In [16]:
filename_parsing_issues = filename_metadata_df.loc[
    (~filename_metadata_df["filename_matched"])
    | filename_metadata_df["campaign_id"].isna()
    | filename_metadata_df["campaign_date"].isna()
    | (
        filename_metadata_df["discovered_file_type"]
        != filename_metadata_df["file_type"]
    ),
    [
        "file",
        "discovered_file_type",
        "campaign_id",
        "campaign_date",
        "file_type",
        "parse_warning",
    ],
]

display(filename_parsing_issues)

,file,discovered_file_type,campaign_id,campaign_date,file_type,parse_warning


## Check campaign file pairing

In [17]:
campaign_file_counts = (
    filename_metadata_df
    .groupby(
        ["campaign_id", "file_type"],
        dropna=False,
    )
    .size()
    .unstack(
        fill_value=0,
    )
    .reset_index()
)

display(campaign_file_counts)

file_type,campaign_id,roster,trades
0,33,1,1
1,34,1,1
2,35,1,1
3,36,1,1
4,37,1,1
5,38,1,1
6,39,1,1
7,40,1,1
8,41,1,1
9,42,1,1


# Normalize column names

In [18]:
def normalize_column_name(column: object) -> str:
    """Converts a raw column name to lowercase snake case.

    Handles camel case, Pascal case, spaces, punctuation, and repeated
    underscores.

    Args:
        column: Original column name.

    Returns:
        The normalized snake-case column name.
    """
    column_name = str(column).strip()

    column_name = re.sub(
        r"([A-Z]+)([A-Z][a-z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )

    column_name = re.sub(
        r"_+",
        "_",
        column_name,
    )

    return column_name.strip("_").lower()

In [19]:
print(normalize_column_name("IP ADDRESS"))
print(normalize_column_name("Telegram Username"))
print(normalize_column_name("accountId"))
print(normalize_column_name("openDateTime"))
print(normalize_column_name("netProfit"))

ip_address
telegram_username
account_id
open_date_time
net_profit


# Load and standardize roster file

## Define roster schema

In [20]:
ROSTER_REQUIRED_COLUMNS = {
    "account_id",
    "email",
    "ip_address",
}

ROSTER_OPTIONAL_COLUMNS = {
    "telegram_username",
    "challenge_type_id",
}

## Detect embedded roster headers

In [21]:
def find_roster_header_echoes(
    df: pd.DataFrame,
) -> pd.Series:
    """Identifies embedded duplicate-header rows in a roster DataFrame.

    A row is treated as an embedded header when the account, email, or
    IP-address field repeats its corresponding column name. Missing values
    are treated as non-matches and are not removed.

    Args:
        df: Standardized roster DataFrame.

    Returns:
        A Boolean Series where `True` identifies an embedded duplicate-header
        row and `False` identifies a normal data row.
    """
    account_echo = (
        df["account_id"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "account",
            "account_id",
            "accountid",
        })
        .fillna(False)
    )

    email_echo = (
        df["email"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("email")
        .fillna(False)
    )

    ip_echo = (
        df["ip_address"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "ip_address",
            "ip address",
        })
        .fillna(False)
    )

    header_echo_mask = (
        account_echo
        | email_echo
        | ip_echo
    )

    return header_echo_mask.astype(bool)

## Define load_roster_file()

In [22]:
def load_roster_file(path: str | Path) -> tuple[pd.DataFrame, dict]:
    """Loads and standardizes one campaign roster file.

    The function reads a CSV or Excel roster file, normalizes its column
    names, validates the required schema, removes embedded duplicate-header
    rows, and attaches campaign metadata extracted from the filename.

    Args:
        path: Path to the roster CSV or Excel file.

    Returns:
        A tuple containing:
            - A standardized roster DataFrame.
            - A dictionary summarizing the loading and cleaning results.

    Raises:
        ValueError: If the file type is unsupported or required columns
            are missing.
    """
    path = Path(path)
    metadata = parse_filename(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

    else:
        raise ValueError(
            f"Unsupported roster file type: {path.suffix}"
        )

    original_row_count = len(df)
    original_columns = df.columns.tolist()

    # Preserve the row location from the source file.
    df.insert(
        loc=0,
        column="source_row_number",
        value=np.arange(2, len(df) + 2),
    )

    # Standardize all source column names.
    df.columns = [
        normalize_column_name(column)
        for column in df.columns
    ]
    
    df = df.rename(
        columns={
            "account": "account_id",
            "accountid": "account_id",
        }
    )

    missing_required_columns = (
        ROSTER_REQUIRED_COLUMNS - set(df.columns)
    )

    if missing_required_columns:
        raise ValueError(
            f"{path.name} is missing required roster columns: "
            f"{sorted(missing_required_columns)}"
        )

    # Add optional columns when they are absent.
    for column in ROSTER_OPTIONAL_COLUMNS:
        if column not in df.columns:
            df[column] = pd.NA

    header_echo_mask = find_roster_header_echoes(df)
    
    if header_echo_mask.isna().any():
        raise RuntimeError(
            f"Header mask contains missing values for {path.name}."
        )
    
    embedded_header_rows = (
        df.loc[header_echo_mask, "source_row_number"]
        .astype(int)
        .tolist()
    )

    df = df.loc[~header_echo_mask].copy()
    
    rows_removed = original_row_count - len(df)

    if rows_removed != len(embedded_header_rows):
        raise RuntimeError(
            f"Row-removal mismatch in {path.name}: "
            f"{rows_removed} rows were removed, but only "
            f"{len(embedded_header_rows)} embedded-header rows "
            f"were recorded."
        )

    # Standardize string-like fields.
    string_columns = [
        "account_id",
        "email",
        "ip_address",
        "telegram_username",
        "challenge_type_id",
    ]

    for column in string_columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

        df[column] = df[column].replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            }
        )

    # Attach source and campaign metadata.
    df["campaign_id"] = metadata["campaign_id"]
    df["campaign_date"] = metadata["campaign_date"]
    df["source_file"] = path.name
    df["source_path"] = str(path)

    standardized_columns = [
        "campaign_id",
        "campaign_date",
        "account_id",
        "email",
        "ip_address",
        "telegram_username",
        "challenge_type_id",
        "source_file",
        "source_path",
        "source_row_number",
    ]

    df = (
        df[standardized_columns]
        .reset_index(drop=True)
    )

    loading_report = {
        "file": path.name,
        "campaign_id": metadata["campaign_id"],
        "campaign_date": metadata["campaign_date"],
        "original_rows": original_row_count,
        "cleaned_rows": len(df),
        "rows_removed": rows_removed,
        "embedded_header_rows_removed": len(embedded_header_rows),
        "embedded_header_row_numbers": embedded_header_rows,
        "original_columns": original_columns,
        "missing_account": int(df["account_id"].isna().sum()),
        "missing_email": int(df["email"].isna().sum()),
        "missing_ip_address": int(df["ip_address"].isna().sum()),
        "error": None,
    }

    return df, loading_report

## Test the first roster file

In [23]:
first_roster_path = source_files.loc[
    source_files["file_type"] == "roster",
    "path",
].iloc[0]

print(first_roster_path)

/Users/charltonsiaw/Desktop/C22-veNTUre/data/User Data/Campaign 33 Data 24 Feb 2026 Traders only (1D).xlsx


In [24]:
test_roster, test_roster_report = load_roster_file(
    first_roster_path
)

In [25]:
print("Shape:", test_roster.shape)
print("Columns:", test_roster.columns.tolist())

display(test_roster.head())
display(pd.Series(test_roster_report).to_frame("value"))

Shape: (500, 10)
Columns: ['campaign_id', 'campaign_date', 'account_id', 'email', 'ip_address', 'telegram_username', 'challenge_type_id', 'source_file', 'source_path', 'source_row_number']


,campaign_id,campaign_date,account_id,email,ip_address,telegram_username,challenge_type_id,source_file,source_path,source_row_number
0,33,2026-02-24,D#1614601,jahanzaibkhalid154@gmail.com,182.185.152.215,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,2
1,33,2026-02-24,D#1702708,muh.irwanto99@gmail.com,13.215.129.254,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,3
2,33,2026-02-24,D#1702434,xbiswas598@gmail.com,77.111.246.27,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,4
3,33,2026-02-24,D#1702729,Wachirajames817@gmail.com,102.210.40.114,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,5
4,33,2026-02-24,D#1702577,naziahell988@gmail.com,37.111.148.168,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...,6


,value
file,Campaign 33 Data 24 Feb 2026 Traders only (1D)...
campaign_id,33
campaign_date,2026-02-24 00:00:00
original_rows,500
cleaned_rows,500
rows_removed,0
embedded_header_rows_removed,0
embedded_header_row_numbers,[]
original_columns,"[ACCOUNT, EMAIL, IP ADDRESS]"
missing_account,0


In [26]:
display(
    test_roster.dtypes
    .astype(str)
    .rename("dtype")
    .to_frame()
)

,dtype
campaign_id,int64
campaign_date,datetime64[us]
account_id,string
email,string
ip_address,string
telegram_username,string
challenge_type_id,string
source_file,str
source_path,str
source_row_number,int64


## Test campaign 57

In [27]:
campaign_57_roster_path = source_files.loc[
    (source_files["file_type"] == "roster")
    & source_files["file"].str.contains(
        r"Campaign[\s_]*57",
        case=False,
        regex=True,
    ),
    "path",
].iloc[0]

campaign_57_roster, campaign_57_report = (
    load_roster_file(campaign_57_roster_path)
)

display(
    pd.Series(campaign_57_report).to_frame("value")
)

,value
file,Campaign 57 Data 02 June 2026 Traders only (1D...
campaign_id,57
campaign_date,2026-06-02 00:00:00
original_rows,504
cleaned_rows,500
rows_removed,4
embedded_header_rows_removed,4
embedded_header_row_numbers,"[102, 203, 304, 405]"
original_columns,"[ACCOUNT, EMAIL, IP ADDRESS]"
missing_account,0


In [28]:
print(
    "Embedded headers removed:",
    campaign_57_report[
        "embedded_header_rows_removed"
    ],
)

print(
    "Source row numbers:",
    campaign_57_report[
        "embedded_header_row_numbers"
    ],
)

Embedded headers removed: 4
Source row numbers: [102, 203, 304, 405]


# Load and standardize trade files

## Define trade schema

In [29]:
TRADE_ID_COLUMNS = [
    "account_id",
    "close_trade_id",
    "position_id",
    "close_order_id",
    "open_order_id",
    "user_group_id",
]

TRADE_DATETIME_COLUMNS = [
    "open_date_time",
    "close_date_time",
]

TRADE_NUMERIC_COLUMNS = [
    "lot_size",
    "duration_sec",
    "profit",
    "reverse_profit",
    "net_profit",
    "commission",
    "swap",
    "amount",
    "open_price",
    "close_price",
    "sl_price",
    "tp_price",
    "open_trade_cross_price",
    "close_trade_cross_price",
]

In [30]:
TRADE_COLUMN_ALIASES = {
    "account_id": "account_id",
    "accountid": "account_id",

    "close_trade_id": "close_trade_id",
    "closetradeid": "close_trade_id",

    "position_id": "position_id",
    "positionid": "position_id",

    "close_order_id": "close_order_id",
    "closeorderid": "close_order_id",

    "open_order_id": "open_order_id",
    "openorderid": "open_order_id",

    "user_group_id": "user_group_id",
    "usergroupid": "user_group_id",

    "open_date_time": "open_date_time",
    "opendatetime": "open_date_time",

    "close_date_time": "close_date_time",
    "closedatetime": "close_date_time",

    "lot_size": "lot_size",
    "lotsize": "lot_size",

    "duration_sec": "duration_sec",
    "durationsec": "duration_sec",

    "profit": "profit",

    "reverse_profit": "reverse_profit",
    "reverseprofit": "reverse_profit",

    "net_profit": "net_profit",
    "netprofit": "net_profit",

    "commission": "commission",
    "swap": "swap",
    "amount": "amount",

    "open_price": "open_price",
    "openprice": "open_price",

    "close_price": "close_price",
    "closeprice": "close_price",

    "sl_price": "sl_price",
    "slprice": "sl_price",

    "tp_price": "tp_price",
    "tpprice": "tp_price",

    "open_trade_cross_price": "open_trade_cross_price",
    "opentradecrossprice": "open_trade_cross_price",

    "close_trade_cross_price": "close_trade_cross_price",
    "closetradecrossprice": "close_trade_cross_price",

    "side": "side",
    "currency": "currency",

    "campaign_id": "source_campaign_id",
    "campaignid": "source_campaign_id",
}

## Load one trade file function

In [31]:
def load_trade_file(path: str | Path) -> tuple[pd.DataFrame, dict]:
    """Loads and standardizes one campaign trade file.

    Args:
        path: Path to a CSV or Excel trade file.

    Returns:
        A tuple containing:
            - A standardized trade DataFrame.
            - A dictionary containing loading and validation information.

    Raises:
        ValueError: If the file type is unsupported or required columns
            are absent.
    """
    path = Path(path)
    metadata = parse_filename(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    elif path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        raise ValueError(
            f"Unsupported trade file type: {path.suffix}"
        )

    original_rows = len(df)
    original_columns = df.columns.tolist()

    normalized_columns = {
        column: normalize_column_name(column)
        for column in df.columns
    }

    df = df.rename(columns=normalized_columns)
    df = df.rename(columns=TRADE_COLUMN_ALIASES)

    df.insert(
        0,
        "source_row_number",
        np.arange(2, len(df) + 2),
    )

    required_columns = {
        "account_id",
        "open_date_time",
        "close_date_time",
        "amount",
        "net_profit",
    }

    missing_required = required_columns - set(df.columns)

    if missing_required:
        raise ValueError(
            f"{path.name} is missing required trade columns: "
            f"{sorted(missing_required)}"
        )

    # Remove actual embedded duplicate headers only.
    header_echo_mask = (
        df["account_id"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "account_id",
            "accountid",
            "account",
        })
        .fillna(False)
        |
        df["open_date_time"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "open_date_time",
            "opendatetime",
        })
        .fillna(False)
    ).astype(bool)

    embedded_header_rows = (
        df.loc[header_echo_mask, "source_row_number"]
        .astype(int)
        .tolist()
    )

    df = df.loc[~header_echo_mask].copy()

    rows_removed = original_rows - len(df)

    if rows_removed != len(embedded_header_rows):
        raise RuntimeError(
            f"Row-removal mismatch in {path.name}: "
            f"{rows_removed} rows removed but "
            f"{len(embedded_header_rows)} recorded."
        )

    for column in TRADE_ID_COLUMNS:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace({
                    "": pd.NA,
                    "nan": pd.NA,
                    "None": pd.NA,
                })
            )

    invalid_numeric_counts = {}

    for column in TRADE_NUMERIC_COLUMNS:
        if column not in df.columns:
            continue

        original_non_missing = df[column].notna()

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

        invalid_numeric_counts[column] = int(
            (
                original_non_missing
                & df[column].isna()
            ).sum()
        )

    invalid_datetime_counts = {}

    for column in TRADE_DATETIME_COLUMNS:
        if column not in df.columns:
            continue

        original_non_missing = df[column].notna()

        df[column] = pd.to_datetime(
            df[column],
            errors="coerce",
            utc=True,
        )

        invalid_datetime_counts[column] = int(
            (
                original_non_missing
                & df[column].isna()
            ).sum()
        )

    if "side" in df.columns:
        df["side"] = (
            df["side"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

    if "currency" in df.columns:
        df["currency"] = (
            df["currency"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    df["campaign_id"] = metadata["campaign_id"]
    df["campaign_date"] = metadata["campaign_date"]
    df["source_file"] = path.name
    df["source_path"] = str(path)

    loading_report = {
        "file": path.name,
        "campaign_id": metadata["campaign_id"],
        "campaign_date": metadata["campaign_date"],
        "original_rows": original_rows,
        "cleaned_rows": len(df),
        "rows_removed": rows_removed,
        "embedded_header_row_numbers": embedded_header_rows,
        "original_columns": original_columns,
        "invalid_numeric_counts": invalid_numeric_counts,
        "invalid_datetime_counts": invalid_datetime_counts,
        "missing_account_id": int(df["account_id"].isna().sum()),
        "missing_open_datetime": int(
            df["open_date_time"].isna().sum()
        ),
        "missing_close_datetime": int(
            df["close_date_time"].isna().sum()
        ),
        "error": None,
    }

    return df.reset_index(drop=True), loading_report

## Test one trade file

In [32]:
first_trade_path = source_files.loc[
    source_files["file_type"] == "trades",
    "path",
].iloc[0]

test_trades, test_trade_report = load_trade_file(
    first_trade_path
)

print("Shape:", test_trades.shape)
print("Columns:", test_trades.columns.tolist())

display(test_trades.head())
display(pd.Series(test_trade_report).to_frame("value"))

Shape: (749, 30)
Columns: ['source_row_number', 'account_id', 'instrument', 'lot_size', 'close_trade_id', 'position_id', 'close_order_id', 'open_order_id', 'duration_sec', 'open_date_time', 'close_date_time', 'profit', 'reverse_profit', 'net_profit', 'commission', 'swap', 'amount', 'open_price', 'close_price', 'sl_price', 'tp_price', 'side', 'currency', 'open_trade_cross_price', 'close_trade_cross_price', 'user_group_id', 'campaign_id', 'campaign_date', 'source_file', 'source_path']


,source_row_number,account_id,instrument,lot_size,close_trade_id,position_id,close_order_id,open_order_id,duration_sec,open_date_time,...,tp_price,side,currency,open_trade_cross_price,close_trade_cross_price,user_group_id,campaign_id,campaign_date,source_file,source_path
0,2,D#1645625,XAUUSD,100,7349874591903930241,7349874591885628592,7349874591964660940,7349874591964660912,631,2026-02-24 00:49:38+00:00,...,5203.46,sell,USD,1,1,1583664,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...
1,3,D#1702379,XAUUSD,100,7349874591903930511,7349874591885628953,7349874591964663694,7349874591964663408,19,2026-02-24 01:02:19+00:00,...,NaN,buy,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...
2,4,D#1645651,XAUUSD,100,7349874591903930537,7349874591885628946,7349874591964663739,7349874591964663378,29,2026-02-24 01:02:15+00:00,...,NaN,sell,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...
3,5,D#1670922,XAUUSD,100,7349874591903930937,7349874591885629157,7349874591964665237,7349874591964665058,38,2026-02-24 01:07:03+00:00,...,NaN,sell,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...
4,6,D#1702617,XAUUSD,100,7349874591903931087,7349874591885629132,7349874591964665786,7349874591964664896,208,2026-02-24 01:06:23+00:00,...,NaN,buy,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,/Users/charltonsiaw/Desktop/C22-veNTUre/data/U...


,value
file,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv
campaign_id,33
campaign_date,2026-02-24 00:00:00
original_rows,749
cleaned_rows,749
rows_removed,0
embedded_header_row_numbers,[]
original_columns,"[accountId, instrument, lotSize, closeTradeId,..."
invalid_numeric_counts,"{'lot_size': 0, 'duration_sec': 0, 'profit': 0..."
invalid_datetime_counts,"{'open_date_time': 0, 'close_date_time': 0}"


# Load all roster and trade files

## Load all roster files

In [33]:
roster_frames = []
roster_loading_reports = []

for file_record in source_files.loc[
    source_files["file_type"] == "roster"
].itertuples(index=False):

    try:
        file_df, report = load_roster_file(
            file_record.path
        )

        roster_frames.append(file_df)
        roster_loading_reports.append(report)

    except Exception as exc:
        roster_loading_reports.append({
            "file": file_record.file,
            "error": str(exc),
        })

roster = pd.concat(
    roster_frames,
    ignore_index=True,
)

roster_loading_report_df = pd.DataFrame(
    roster_loading_reports
)

## Load all trade files

In [34]:
trade_frames = []
trade_loading_reports = []

for file_record in source_files.loc[
    source_files["file_type"] == "trades"
].itertuples(index=False):

    try:
        file_df, report = load_trade_file(
            file_record.path
        )

        trade_frames.append(file_df)
        trade_loading_reports.append(report)

    except Exception as exc:
        trade_loading_reports.append({
            "file": file_record.file,
            "error": str(exc),
        })

trades = pd.concat(
    trade_frames,
    ignore_index=True,
)

trade_loading_report_df = pd.DataFrame(
    trade_loading_reports
)

In [35]:
trades = (
    trades
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
        ]
    )
    .reset_index(drop=True)
)

## Check loading errors

In [36]:
print("Roster loading errors:")

display(
    roster_loading_report_df.loc[
        roster_loading_report_df["error"].notna()
    ]
)

print("Trade loading errors:")

display(
    trade_loading_report_df.loc[
        trade_loading_report_df["error"].notna()
    ]
)

Roster loading errors:


,file,campaign_id,campaign_date,original_rows,cleaned_rows,rows_removed,embedded_header_rows_removed,embedded_header_row_numbers,original_columns,missing_account,missing_email,missing_ip_address,error


Trade loading errors:


,file,campaign_id,campaign_date,original_rows,cleaned_rows,rows_removed,embedded_header_row_numbers,original_columns,invalid_numeric_counts,invalid_datetime_counts,missing_account_id,missing_open_datetime,missing_close_datetime,error


# Validate the combined data

## Basic shape

In [37]:
data_summary = pd.Series({
    "roster_rows": len(roster),
    "trade_rows": len(trades),
    "roster_campaigns": roster[
        "campaign_id"
    ].nunique(),
    "trade_campaigns": trades[
        "campaign_id"
    ].nunique(),
    "registered_accounts": roster[
        "account_id"
    ].nunique(),
    "active_accounts": trades[
        "account_id"
    ].nunique(),
})

display(data_summary.to_frame("value"))

,value
roster_rows,15874
trade_rows,46520
roster_campaigns,34
trade_campaigns,34
registered_accounts,500
active_accounts,502


## Build unique roster and active account-campaign keys

In [38]:
roster_account_campaigns = (
    roster[
        ["campaign_id", "account_id"]
    ]
    .dropna(
        subset=["campaign_id", "account_id"]
    )
    .drop_duplicates()
)

active_account_campaigns = (
    trades[
        ["campaign_id", "account_id"]
    ]
    .dropna(
        subset=["campaign_id", "account_id"]
    )
    .drop_duplicates()
)

In [39]:
print(
    "Registered account-campaign combinations:",
    len(roster_account_campaigns),
)

print(
    "Active account-campaign combinations:",
    len(active_account_campaigns),
)

Registered account-campaign combinations: 15874
Active account-campaign combinations: 8165


## Find the campaigns where active exceeds registered

In [40]:
registered_by_campaign = (
    roster_account_campaigns
    .groupby("campaign_id")
    .size()
    .rename("registered_accounts")
)

active_by_campaign = (
    active_account_campaigns
    .groupby("campaign_id")
    .size()
    .rename("active_accounts")
)

trade_rows_by_campaign = (
    trades.groupby("campaign_id")
    .size()
    .rename("trade_rows")
)

campaign_summary = pd.concat(
    [
        registered_by_campaign,
        active_by_campaign,
        trade_rows_by_campaign,
    ],
    axis=1,
).reset_index()

In [41]:
count_columns = [
    "registered_accounts",
    "active_accounts",
    "trade_rows",
]

campaign_summary[count_columns] = (
    campaign_summary[count_columns]
    .fillna(0)
    .astype(int)
)

In [42]:
campaign_summary["participation_rate"] = np.where(
    campaign_summary["registered_accounts"] > 0,
    (
        campaign_summary["active_accounts"]
        / campaign_summary["registered_accounts"]
    ),
    np.nan,
)

In [43]:
campaign_summary["active_minus_registered"] = (
    campaign_summary["active_accounts"]
    - campaign_summary["registered_accounts"]
)

In [44]:
display(
    campaign_summary.sort_values("campaign_id")
)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate,active_minus_registered
0,33,500,137,749,0.274000,-363
1,34,281,192,942,0.683274,-89
2,35,500,185,1030,0.370000,-315
3,36,401,221,1457,0.551122,-180
4,37,455,236,1432,0.518681,-219
5,38,500,205,1271,0.410000,-295
6,39,340,205,1180,0.602941,-135
7,40,381,248,1432,0.650919,-133
8,41,223,177,1227,0.793722,-46
9,42,500,269,1674,0.538000,-231


## Check campaigns where active exceeds registered

In [45]:
campaigns_active_exceeds_registered = (
    campaign_summary.loc[
        campaign_summary["active_accounts"]
        > campaign_summary["registered_accounts"]
    ]
    .sort_values("campaign_id")
)

display(campaigns_active_exceeds_registered)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate,active_minus_registered


## Find the exact accounts missing from the roster

In [46]:
active_roster_match = (
    active_account_campaigns
    .merge(
        roster_account_campaigns,
        on=["campaign_id", "account_id"],
        how="left",
        indicator=True,
    )
)

In [47]:
active_not_in_roster = (
    active_roster_match.loc[
        active_roster_match["_merge"] == "left_only",
        ["campaign_id", "account_id"],
    ]
    .sort_values(["campaign_id", "account_id"])
    .reset_index(drop=True)
)

print(
    "Active account-campaigns absent from roster:",
    len(active_not_in_roster),
)

display(active_not_in_roster)

Active account-campaigns absent from roster: 6


,campaign_id,account_id
0,33,D#1645625
1,33,D#1645639
2,33,D#1759507
3,34,D#1702452
4,37,D#1702495
5,37,D#1702657


In [48]:
active_not_in_roster_summary = (
    active_not_in_roster
    .groupby("campaign_id")
    .size()
    .rename("active_not_in_roster")
    .reset_index()
)

display(active_not_in_roster_summary)

,campaign_id,active_not_in_roster
0,33,3
1,34,1
2,37,2


In [49]:
campaign_summary = campaign_summary.merge(
    active_not_in_roster_summary,
    on="campaign_id",
    how="left",
)

campaign_summary["active_not_in_roster"] = (
    campaign_summary["active_not_in_roster"]
    .fillna(0)
    .astype(int)
)

display(
    campaign_summary.sort_values("campaign_id")
)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate,active_minus_registered,active_not_in_roster
0,33,500,137,749,0.274000,-363,3
1,34,281,192,942,0.683274,-89,1
2,35,500,185,1030,0.370000,-315,0
3,36,401,221,1457,0.551122,-180,0
4,37,455,236,1432,0.518681,-219,2
5,38,500,205,1271,0.410000,-295,0
6,39,340,205,1180,0.602941,-135,0
7,40,381,248,1432,0.650919,-133,0
8,41,223,177,1227,0.793722,-46,0
9,42,500,269,1674,0.538000,-231,0


## Find registered accounts that never traded

In [50]:
roster_active_match = (
    roster_account_campaigns
    .merge(
        active_account_campaigns,
        on=["campaign_id", "account_id"],
        how="left",
        indicator=True,
    )
)

registered_not_active = (
    roster_active_match.loc[
        roster_active_match["_merge"] == "left_only",
        ["campaign_id", "account_id"],
    ]
    .sort_values(["campaign_id", "account_id"])
    .reset_index(drop=True)
)

print(
    "Registered account-campaigns with no trade:",
    len(registered_not_active),
)

display(registered_not_active.head(20))

Registered account-campaigns with no trade: 7715


,campaign_id,account_id
0,33,D#1589383
1,33,D#1589404
2,33,D#1589405
3,33,D#1589407
4,33,D#1589409
5,33,D#1589424
6,33,D#1589450
7,33,D#1589455
8,33,D#1589456
9,33,D#1589457


In [51]:
registered_not_active_summary = (
    registered_not_active
    .groupby("campaign_id")
    .size()
    .rename("registered_not_active")
    .reset_index()
)

campaign_summary = campaign_summary.merge(
    registered_not_active_summary,
    on="campaign_id",
    how="left",
)

campaign_summary["registered_not_active"] = (
    campaign_summary["registered_not_active"]
    .fillna(0)
    .astype(int)
)

In [52]:
display(
    campaign_summary[
        [
            "campaign_id",
            "registered_accounts",
            "active_accounts",
            "active_not_in_roster",
            "registered_not_active",
            "trade_rows",
            "participation_rate",
            "active_minus_registered",
        ]
    ].sort_values("campaign_id")
)

,campaign_id,registered_accounts,active_accounts,active_not_in_roster,registered_not_active,trade_rows,participation_rate,active_minus_registered
0,33,500,137,3,366,749,0.274000,-363
1,34,281,192,1,90,942,0.683274,-89
2,35,500,185,0,315,1030,0.370000,-315
3,36,401,221,0,180,1457,0.551122,-180
4,37,455,236,2,221,1432,0.518681,-219
5,38,500,205,0,295,1271,0.410000,-295
6,39,340,205,0,135,1180,0.602941,-135
7,40,381,248,0,133,1432,0.650919,-133
8,41,223,177,0,46,1227,0.793722,-46
9,42,500,269,0,231,1674,0.538000,-231


In [53]:
data_summary = pd.Series({
    "roster_rows": len(roster),
    "trade_rows": len(trades),
    "roster_campaigns": roster["campaign_id"].nunique(),
    "trade_campaigns": trades["campaign_id"].nunique(),
    "unique_registered_account_ids": (
        roster["account_id"].nunique()
    ),
    "unique_active_account_ids": (
        trades["account_id"].nunique()
    ),
    "registered_account_campaigns": len(
        roster_account_campaigns
    ),
    "active_account_campaigns": len(
        active_account_campaigns
    ),
    "active_account_campaigns_not_in_roster": len(
        active_not_in_roster
    ),
})

display(data_summary.to_frame("value"))

,value
roster_rows,15874
trade_rows,46520
roster_campaigns,34
trade_campaigns,34
unique_registered_account_ids,500
unique_active_account_ids,502
registered_account_campaigns,15874
active_account_campaigns,8165
active_account_campaigns_not_in_roster,6


## Find registered-only and active-only account IDs

In [54]:
registered_account_ids = set(
    roster["account_id"].dropna().unique()
)

active_account_ids = set(
    trades["account_id"].dropna().unique()
)

active_only_account_ids = sorted(
    active_account_ids - registered_account_ids
)

registered_only_account_ids = sorted(
    registered_account_ids - active_account_ids
)

print("Active-only account IDs:", active_only_account_ids)
print("Count:", len(active_only_account_ids))

print("\nRegistered-only account IDs:", registered_only_account_ids)
print("Count:", len(registered_only_account_ids))

Active-only account IDs: ['D#1645625', 'D#1645639', 'D#1759507']
Count: 3

Registered-only account IDs: ['D#1589383']
Count: 1


## Campaign-by-campaign counts

In [55]:
campaign_summary = (
    roster.groupby("campaign_id")
    .agg(
        registered_accounts=(
            "account_id",
            "nunique",
        )
    )
    .join(
        trades.groupby("campaign_id").agg(
            active_accounts=(
                "account_id",
                "nunique",
            ),
            trade_rows=(
                "account_id",
                "size",
            ),
        ),
        how="outer",
    )
    .reset_index()
)

campaign_summary["participation_rate"] = (
    campaign_summary["active_accounts"]
    / campaign_summary["registered_accounts"]
)

display(campaign_summary)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate
0,33,500,137,749,0.274000
1,34,281,192,942,0.683274
2,35,500,185,1030,0.370000
3,36,401,221,1457,0.551122
4,37,455,236,1432,0.518681
5,38,500,205,1271,0.410000
6,39,340,205,1180,0.602941
7,40,381,248,1432,0.650919
8,41,223,177,1227,0.793722
9,42,500,269,1674,0.538000


## Check whether active traders exist in the roster

In [56]:
trade_roster_check = trades[
    ["campaign_id", "account_id"]
].drop_duplicates().merge(
    roster[
        ["campaign_id", "account_id"]
    ].drop_duplicates(),
    on=["campaign_id", "account_id"],
    how="left",
    indicator=True,
)

active_not_in_roster = trade_roster_check.loc[
    trade_roster_check["_merge"] == "left_only"
]

print(
    "Active account-campaigns not found in roster:",
    len(active_not_in_roster),
)

display(active_not_in_roster.head(20))

Active account-campaigns not found in roster: 6


,campaign_id,account_id,_merge
8,33,D#1645625,left_only
9,33,D#1645639,left_only
136,33,D#1759507,left_only
233,34,D#1702452,left_only
865,37,D#1702495,left_only
933,37,D#1702657,left_only


## Check trader durations

In [57]:
trades["calculated_duration_sec"] = (
    trades["close_date_time"]
    - trades["open_date_time"]
).dt.total_seconds()

In [58]:
duration_validation = pd.Series({
    "missing_open_datetime": trades[
        "open_date_time"
    ].isna().sum(),
    "missing_close_datetime": trades[
        "close_date_time"
    ].isna().sum(),
    "negative_duration": (
        trades["calculated_duration_sec"] < 0
    ).sum(),
    "zero_duration": (
        trades["calculated_duration_sec"] == 0
    ).sum(),
})

display(duration_validation.to_frame("value"))

,value
missing_open_datetime,0
missing_close_datetime,0
negative_duration,0
zero_duration,13


## Validate P&L reconciliation

In [59]:
required_pnl_columns = {
    "net_profit",
    "profit",
    "commission",
    "swap",
}

if required_pnl_columns.issubset(trades.columns):
    trades["expected_net_profit"] = (
        trades["profit"].fillna(0)
        + trades["commission"].fillna(0)
        + trades["swap"].fillna(0)
    )

    trades["net_profit_difference"] = (
        trades["net_profit"]
        - trades["expected_net_profit"]
    )

    trades["net_profit_reconciles"] = (
        trades["net_profit_difference"].abs()
        <= 0.01
    )

    print(
        "Rows failing P&L reconciliation:",
        (
            ~trades["net_profit_reconciles"]
        ).sum(),
    )

Rows failing P&L reconciliation: 0


## Check identifier duplication

In [60]:
identifier_checks = {}

for column in [
    "close_trade_id",
    "open_order_id",
    "close_order_id",
    "position_id",
]:
    if column in trades.columns:
        identifier_checks[
            f"duplicate_{column}"
        ] = trades.duplicated(
            subset=["campaign_id", column],
            keep=False,
        ).sum()

display(
    pd.Series(identifier_checks).to_frame("value")
)

,value
duplicate_close_trade_id,3138
duplicate_open_order_id,3138
duplicate_close_order_id,3138
duplicate_position_id,3138


# Build trade features

## Attach previous completed trade

In [61]:
def attach_previous_completed_trade(
    trades: pd.DataFrame,
) -> pd.DataFrame:
    """Attaches the most recently completed prior trade to each trade.

    Only trades whose close time occurred on or before the current trade's
    open time are eligible. This prevents unresolved or overlapping trades
    from leaking future outcomes into behavioral features.

    Args:
        trades: Standardized trade-level DataFrame.

    Returns:
        A copy of the trade DataFrame with previous-completed-trade
        information attached.
    """
    result_frames = []

    grouping_columns = [
        "campaign_id",
        "account_id",
    ]

    for _, group in trades.groupby(
        grouping_columns,
        dropna=False,
        sort=False,
    ):
        current = (
            group
            .sort_values("open_date_time")
            .copy()
        )

        completed = (
            group.loc[
                group["close_date_time"].notna()
            ]
            .sort_values("close_date_time")
            .copy()
        )

        previous_columns = [
            "close_date_time",
            "net_profit",
            "amount",
            "position_id",
        ]

        previous_columns = [
            column
            for column in previous_columns
            if column in completed.columns
        ]

        previous = completed[
            previous_columns
        ].rename(
            columns={
                "close_date_time":
                    "previous_completed_close_date_time",
                "net_profit":
                    "previous_completed_net_profit",
                "amount":
                    "previous_completed_amount",
                "position_id":
                    "previous_completed_position_id",
            }
        )

        merged = pd.merge_asof(
            current.sort_values("open_date_time"),
            previous.sort_values(
                "previous_completed_close_date_time"
            ),
            left_on="open_date_time",
            right_on="previous_completed_close_date_time",
            direction="backward",
            allow_exact_matches=True,
        )

        result_frames.append(merged)

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result["previous_completed_was_loss"] = (
        result["previous_completed_net_profit"] < 0
    )

    result["previous_completed_was_win"] = (
        result["previous_completed_net_profit"] > 0
    )

    result["reentry_gap_minutes"] = (
        result["open_date_time"]
        - result[
            "previous_completed_close_date_time"
        ]
    ).dt.total_seconds() / 60

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
            ]
        )
        .reset_index(drop=True)
    )

In [62]:
trades_with_previous_completed = (
    attach_previous_completed_trade(trades)
)

In [63]:
display(
    trades_with_previous_completed[
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "net_profit",
            "previous_completed_close_date_time",
            "previous_completed_net_profit",
            "previous_completed_was_loss",
            "reentry_gap_minutes",
        ]
    ].head(20)
)

,campaign_id,account_id,open_date_time,close_date_time,net_profit,previous_completed_close_date_time,previous_completed_net_profit,previous_completed_was_loss,reentry_gap_minutes
0,33,D#1589378,2026-02-24 12:01:08+00:00,2026-02-24 12:02:17+00:00,0.01,NaT,NaN,False,NaN
1,33,D#1589408,2026-02-24 02:46:35+00:00,2026-02-24 02:52:47+00:00,-187.92,NaT,NaN,False,NaN
2,33,D#1589408,2026-02-24 02:53:40+00:00,2026-02-24 02:56:49+00:00,5.30,2026-02-24 02:52:47+00:00,-187.92,True,0.883333
3,33,D#1589408,2026-02-24 02:57:37+00:00,2026-02-24 02:58:40+00:00,-72.00,2026-02-24 02:56:49+00:00,5.30,False,0.800000
4,33,D#1589410,2026-02-25 00:13:04+00:00,2026-02-25 00:21:26+00:00,45.40,NaT,NaN,False,NaN
5,33,D#1589410,2026-02-25 00:23:39+00:00,2026-02-25 00:43:42+00:00,42.20,2026-02-25 00:21:26+00:00,45.40,False,2.216667
6,33,D#1589419,2026-02-24 05:32:22+00:00,2026-02-25 00:43:48+00:00,-20.11,NaT,NaN,False,NaN
7,33,D#1589428,2026-02-24 02:38:47+00:00,2026-02-24 02:44:09+00:00,13.00,NaT,NaN,False,NaN
8,33,D#1589428,2026-02-24 07:24:19+00:00,2026-02-24 07:29:48+00:00,108.29,2026-02-24 02:44:09+00:00,13.00,False,280.166667
9,33,D#1589428,2026-02-24 08:38:11+00:00,2026-02-24 12:17:20+00:00,-334.36,2026-02-24 07:29:48+00:00,108.29,False,68.383333


## Validate there is no look-ahead leakage

In [64]:
previous_trade_leakage = (
    trades_with_previous_completed[
        "previous_completed_close_date_time"
    ]
    >
    trades_with_previous_completed[
        "open_date_time"
    ]
)

print(
    "Previous completed trades closing after current entry:",
    previous_trade_leakage.fillna(False).sum(),
)

Previous completed trades closing after current entry: 0


## Verify gaps are not negative

In [65]:
print(
    "Negative re-entry gaps:",
    (
        trades_with_previous_completed[
            "reentry_gap_minutes"
        ] < 0
    ).sum(),
)

Negative re-entry gaps: 0


# Save clean analytical tables

## Create a non-PII roster

In [66]:
trader_roster = (
    roster[
        [
            "campaign_id",
            "campaign_date",
            "account_id",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

## Keep PII separate

In [67]:
trader_identity = (
    roster[
        [
            "account_id",
            "email",
            "ip_address",
            "telegram_username",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

## Save the main output

In [68]:
trades.to_parquet(
    OUTPUT_DIR / "trades_all_campaigns.parquet",
    index=False,
)

trader_roster.to_parquet(
    OUTPUT_DIR / "trader_roster_non_pii.parquet",
    index=False,
)

trades_with_previous_completed.to_parquet(
    OUTPUT_DIR / "trades_with_previous_completed.parquet",
    index=False,
)

campaign_summary.to_csv(
    OUTPUT_DIR / "campaign_summary.csv",
    index=False,
)

roster_loading_report_df.to_csv(
    OUTPUT_DIR / "roster_loading_report.csv",
    index=False,
)

trade_loading_report_df.to_csv(
    OUTPUT_DIR / "trade_loading_report.csv",
    index=False,
)